In [1]:
import io
import os
import cv2 as cv

from PIL import Image
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel

app = FastAPI()


class VideoRequest(BaseModel):
    file_path: str


@app.post("/convert")
def convert(req: VideoRequest):
    # Проверяем, существует ли файл
    if not os.path.exists(req.file_path):
        raise HTTPException(status_code=404, detail="Видео файл не найден")

    # Открываем видео
    cap = cv.VideoCapture(req.file_path)

    # Читаем первый кадр
    ok, firstFrame = cap.read()
    if not ok or firstFrame is None:
        cap.release()
        raise HTTPException(status_code=400, detail="Не удалось прочитать первый кадр видео")

    imgs = []

    # Координаты области
    x1 = 100
    y1 = 50
    x2 = 700
    y2 = 400

    # Проверка координат
    h, w = firstFrame.shape[:2]
    if not (0 <= x1 < x2 <= w and 0 <= y1 < y2 <= h):
        cap.release()
        raise HTTPException(
            status_code=400,
            detail=f"Неверные координаты. Размер кадра: width={w}, height={h}"
        )

    # Показываем первый кадр с рамкой
    preview = firstFrame.copy()
    cv.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv.imshow("Preview", preview)

    print("Проверь рамку")
    print("Нажми любую клавишу, чтобы запустить")
    print("Нажми q, чтобы отменить запуск")

    key = cv.waitKey(0)

    # Отмена
    if key == ord("q"):
        cv.destroyAllWindows()
        cap.release()
        raise HTTPException(status_code=400, detail="Обработка отменена пользователем")

    while True:
        ok, frame = cap.read()
        if not ok or frame is None:
            break

        # Вырезаем область
        slide = frame[y1:y2, x1:x2]

        # Показываем вырезанный участок
        cv.imshow("Slide", slide)
        k = cv.waitKey(30)

        # Сохранение кадра
        if k == ord("s"):
            imgs.append(Image.fromarray(cv.cvtColor(slide, cv.COLOR_BGR2RGB)))
            print("Сохранено кадров:", len(imgs))

        # Выход
        if k == ord("q"):
            break

    cv.destroyAllWindows()
    cap.release()

    if not imgs:
        raise HTTPException(status_code=400, detail="Ни одного кадра не сохранено")

    # Сохраняем PDF в память
    buf = io.BytesIO()
    imgs[0].save(buf, format="PDF", save_all=True, append_images=imgs[1:])
    buf.seek(0)

    return StreamingResponse(
        buf,
        media_type="application/pdf",
        headers={"Content-Disposition": "attachment; filename=slides.pdf"}
    )

In [4]:
import os
from fastapi.testclient import TestClient

client = TestClient(app)

file_path = r"C:\Users\mrafa\Desktop\IMG_7259.MP4"

response = client.post("/convert", json={"file_path": file_path})

print("Status code:", response.status_code)

if response.status_code == 200:
    with open("slides.pdf", "wb") as f:
        f.write(response.content)

    print("Готово: slides.pdf")
    os.startfile("slides.pdf")
else:
    print("Ошибка:")
    try:
        print(response.json())
    except Exception:
        print(response.text)

Проверь рамку
Нажми любую клавишу, чтобы запустить
Нажми q, чтобы отменить запуск
Status code: 400
Ошибка:
{'detail': 'Обработка отменена пользователем'}
